# Создание и заполнение данных БД Postgre

In [1]:
%pip install python-dotenv psycopg2-binary
%pip install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import psycopg2
import pandas as pd
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [3]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [5]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [17]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [18]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


In [10]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура, специалитет, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [104]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


In [8]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost",
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

conn.autocommit = True

Успешное подключение к базе данных!


In [9]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
cursor.execute(query)
rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]
print(f"Колонки: {columns}\n")

for row in rows:
    print(row)

Колонки: ['courseid', 'userid', 'num_week', 's_all', 's_all_avg', 's_course_viewed', 's_course_viewed_avg', 's_q_attempt_viewed', 's_q_attempt_viewed_avg', 's_a_course_module_viewed', 's_a_course_module_viewed_avg', 's_a_submission_status_viewed', 's_a_submission_status_viewed_avg', 'namer_level', 'name_vatt', 'depart', 'name_osno', 'name_formopril', 'leveled', 'num_sem', 'kurs', 'date_vatt']

(84275, 20902, 27, 0, '1', 0, '0,4091', 0, '0', 0, 0.1818, 0, '0.1818', '4', 'Экзамен', 14, '2', '1', '1', 8, 5, '27.06.2022')
(82281, 20857, 26, 0, '0.381', 0, '0,2381', 0, '0', 0, 0.0476, 0, '0.0476', '5', 'Экзамен', 5, '1', '1', '1', 8, 5, '05.07.2022')
(85001, 29636, 11, 0, '0', 0, '0', 0, '0', 0, 0.0, 0, '0', '5', 'Экзамен', 23, '2', '2', '1', 2, 2, '27.06.2022')
(75810, 29171, 26, 0, '2.7619', 0, '0,8095', 0, '0', 0, 0.4286, 0, '0.2381', '4', 'Экзамен', 7, '1', '1', '2', 4, 3, '27.06.2022')
(88349, 32431, 14, 0, '0', 0, '0', 0, '0', 0, 0.0, 0, '0', '3', 'Экзамен', 3, '2', '2', '1', 4, 3, '2

In [21]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_17384\1171977693.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,depart,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt
0,84110,33658,29,0,38.5833,0,"10,4583",0,"2,8333",0,...,6.625,5,Экзамен,11,1,1,1,2,2,17.06.2022
1,73008,23224,10,1,0.2,0,0,0,0,0,...,0,2,Экзамен,41,2,2,1,8,5,23.06.2022
2,78241,36468,14,0,0.1111,0,"0,1111",0,0,0,...,0,4,Экзамен,23,2,2,1,2,2,23.06.2022
3,76411,35687,21,16,17.625,4,"3,125",0,0,6,...,5.5625,2,Экзамен,12,1,1,2,2,2,21.06.2022
4,79426,36753,22,35,2.9412,9,1,0,0,11,...,0.4706,3,Экзамен,7,2,2,2,2,2,24.06.2022
5,72831,33606,11,7,7,3,"3,3333",0,0,0,...,0.8333,4,Экзамен,38,1,1,1,2,2,20.06.2022
6,78352,36328,15,4,10.5,2,"2,9",0,0,1,...,1.8,5,Экзамен,23,2,2,1,2,2,30.06.2022
7,72970,25485,29,0,50.6667,0,"11,4167",0,"8,0417",0,...,11.9583,5,Экзамен,41,2,1,1,6,4,17.06.2022
8,74263,17641,25,13,1.25,1,"0,25",0,0,5,...,0.15,4,Экзамен,5,2,2,1,10,6,21.06.2022
9,71541,27630,27,0,9.9091,0,5,0,0,0,...,1.5,5,Экзамен,20,2,2,1,6,4,27.06.2022


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [38]:
query = """
    SELECT 
        COUNT(DISTINCT depart) AS departments_count
    FROM USER_LOGS
    WHERE courseid IS NOT NULL
"""

query = """
    SELECT
        COUNT(DISTINCT userid)
    FROM USER_LOGS
"""

# query = """ 
#     SELECT 
#         userid,
#         --courseid,
#         num_week,
#         s_all,
#         s_all_avg
        
#     FROM USER_LOGS
#     WHERE userid ='35428' AND courseid = '83989' AND num_week <= 100
#     ORDER BY num_week
# """

# query = """ 
#     SELECT
#         userid,
#         SUM(s_all) / COUNT(num_week) AS avg_events
#     FROM USER_LOGS
#     WHERE userid = '35428' AND courseid = '83989'
#     GROUP BY userid
# """

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_17384\1377096061.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count
0,6316


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводить сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [25]:
query = """
    SELECT 
        DEPARTMENTS.name AS DEPTNAME,
        COUNT(USER_LOGS.courseid) AS COUNTCOURSE
    FROM USER_LOGS INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    GROUP BY DEPARTMENTS.name
    ORDER BY COUNTCOURSE DESC
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_18856\1262581724.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,deptname,countcourse
0,МиХТ,25296
1,ДиСО,25176
2,ЛиП,19008
3,РМПИ,17856
4,ГМДиОПИ,16968
5,ТОМ,16704
6,БИиИТ,16176
7,ПОиД,14760
8,АЭПиМ,14232
9,ЛиУТС,13440


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [32]:
query = """

    SELECT 
        COUNT(DEPTCOUNT.courseid)
    FROM 
    (
        SELECT
            USER_LOGS.courseid,
            COUNT(DISTINCT USER_LOGS.depart) AS departments_count
        FROM USER_LOGS
        WHERE courseid IS NOT NULL
        GROUP BY USER_LOGS.courseid
        HAVING COUNT(DISTINCT depart) > 1
    ) AS DEPTCOUNT 
"""



df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\866560846.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count
0,60


In [42]:
#получили курсы, на которые закреплены несколько кафедр
query = """
    SELECT
        USER_LOGS.courseid AS COURSEID,
        COUNT(DISTINCT USER_LOGS.depart) AS DEPCOUNT
    FROM
        USER_LOGS
    GROUP BY USER_LOGS.courseid
    HAVING COUNT(DISTINCT USER_LOGS.depart) > 1
"""

#мы должны теперь найти уникальные кафедры, сравнивая их курсы с курсами, имеющими несколько кафедр
#тем самым на каждый курс получим кафедры, которые закреплены именно на один курс.
query = """

    WITH COURSES AS 
    (
	SELECT
		USER_LOGS.courseid AS COURSEID,
		COUNT(DISTINCT USER_LOGS.depart) AS DEPCOUNT
	FROM
		USER_LOGS
	GROUP BY USER_LOGS.courseid
	HAVING COUNT(DISTINCT USER_LOGS.depart) > 1
    )

    SELECT DISTINCT
	    DEPARTMENTS.name,
	    UL.courseid
    FROM USER_LOGS AS UL
	    INNER JOIN DEPARTMENTS ON UL.DEPART = DEPARTMENTS.ID
	    INNER JOIN COURSES ON UL.COURSEID = COURSES.COURSEID
    ORDER BY UL.courseid
"""


df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_18856\461634636.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,name,courseid
0,ГМиТТК,71495
1,ПиСЗ,71495
2,УиИС,71495
3,ПиСЗ,71508
4,УиИС,71508
...,...,...
165,ПМиИ,88653
166,БИиИТ,88659
167,ПМиИ,88659
168,ЛиП,88888


## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [51]:
conn.rollback()

In [37]:
query = """
    SELECT 
        namer_level AS namer_level,
        COUNT(DISTINCT userid) AS count
    FROM user_logs
    GROUP BY namer_level
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_17384\1613593422.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,namer_level,count
0,2,1069
1,3,1884
2,4,3243
3,5,3407


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за весь период обучения).

In [28]:
conn.rollback()

In [46]:



query = """

        SELECT 
            userid,
            AVG(namer_level::INT),
            SUM(s_all) AS total_events
        FROM USER_LOGS
        GROUP BY userid
        ORDER BY total_events DESC
        LIMIT 1
   

"""

# query=""" 
#     SELECT 
#             userid,
#             SUM(s_all)
#         FROM USER_LOGS
#         GROUP BY userid
# """

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_17384\1260339353.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,userid,avg,total_events
0,28871,5.0,10300


## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [ ]:
conn.rollback()

In [32]:
query = """
    SELECT
        num_week,
        AVG(s_all)
    FROM USER_LOGS
    GROUP BY num_week
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\3882656882.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,num_week,avg
0,6,13.796607
1,7,9.616142
2,8,8.028543
3,9,9.393296
4,10,8.208546
5,11,10.022001
6,12,9.381716
7,13,10.014011
8,14,9.860178
9,15,10.353694


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [78]:
conn.rollback()

In [80]:
query = """
    SELECT
        COUNT(DISTINCT USER_LOGS.userid) AS count_students,
        DEPARTMENTS.name
    FROM USER_LOGS
        INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    WHERE namer_level = '5'
    GROUP BY DEPARTMENTS.name
    ORDER BY count_students DESC
    LIMIT 1
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\2177089847.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count_students,name
0,310,ДиСО


In [79]:
#Список кафедр и количество студентов в каждой из них
query = """

    SELECT
        COUNT(DISTINCT userid),
        depart
    FROM USER_LOGS
    WHERE namer_level = '2'
    GROUP BY depart
"""

query = """
    SELECT
        COUNT(DISTINCT USER_LOGS.userid) AS count_students,
        DEPARTMENTS.name
    FROM USER_LOGS
        INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    WHERE namer_level = '2'
    GROUP BY DEPARTMENTS.name
    ORDER BY count_students DESC
    LIMIT 1
"""



df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\270051105.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count_students,name
0,72,Эконом.


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [22]:
conn.rollback()

In [23]:
#conn.rollback()

query = """
    SELECT
        name_osno,
        week_number,
        avg_s_all,
        avg_s_course_viewed,
        avg_s_q_attempt_viewed
    FROM USER_LOGS
    WHERE namer_level = '2'
    GROUP BY depart
"""

#получили самую насыщенную по событиям неделю
query = """
    SELECT
        num_week AS week_number,
        SUM(s_all) AS week_events
    FROM USER_LOGS
    GROUP BY num_week
    ORDER BY SUM(s_all) DESC
    LIMIT 1
"""

query = """
    SELECT
        name_osno,
        num_week,
        AVG(s_all_avg::REAL) AS avg_all_action,
        AVG(s_a_course_module_viewed_avg::REAL) AS avg_module_view,
        AVG(s_a_submission_status_viewed_avg::REAL) AS avg_sent_answers
    FROM USER_LOGS
    WHERE num_week = 24
    GROUP BY name_osno, num_week
    
"""




df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16100\1212247874.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,name_osno,num_week,avg_all_action,avg_module_view,avg_sent_answers
0,1,24,13.421515,2.369634,1.891641
1,2,24,10.798811,1.876270,1.450890
